In [ ]:
import random
import copy

class Node:
    def __init__(self, floor_state, position: tuple, parent, birth_action):
        self.floor_state = floor_state
        self.position = position
        self.parent = parent
        self.birth_action = birth_action
        self.path_cost = 0 #số ô bẩn sô với GOAL
        self.heuristic_cost = 0 #khoảng cách manhattan
        self.total_cost = 0
        
    def cal_total_cost(self):  
        self.total_cost = self.path_cost + self.heuristic_cost
        
    def get_tuple_floor_state(self):
        lists = []
        for index in range(NUMBER_OF_STARTED_STATE):
            lists.append(tuple(tuple(row) for row in self.floor_state[index]))
        return tuple(state for state in lists)
    
class Queue:
    def __init__(self):
        self.queue = []

    def is_empty(self):
        return len(self.queue) == 0
    
    def enqueue(self, node):
        self.queue.append(node)
        
    def dequeue(self) -> Node:
        return self.queue.pop(0)
    
    def pop(self, index: int):
        return self.queue.pop(index)
        
    def pop_the_highest_priority(self) -> Node:
        # Tìm node có tổng chi phí (total_cost) nhỏ nhất
        max_priority_index = 0
        for i in range(len(self.queue)):
            if self.queue[i].total_cost < self.queue[max_priority_index].total_cost:
                max_priority_index = i
        return self.queue.pop(max_priority_index)
    
    def contain(self, other: Node):
        for i, node in enumerate(self.queue):
            if node.get_tuple_floor_state() == other.get_tuple_floor_state() and node.position == other.position:
                return i
        return -1

GOAL_STATE = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
ROW, COL = 3, 3
NUMBER_OF_STARTED_STATE = 2

def floor_and_vacuumpos_initialize():
    floor = []
    vacuum_pos = (random.randint(0, ROW - 1), random.randint(0, COL - 1))
    for i in range(ROW):
        floor.append([random.randint(0, 1) for _ in range(COL)])
    if floor[vacuum_pos[0]][vacuum_pos[1]] == 1:
        floor[vacuum_pos[0]][vacuum_pos[1]] = 0
    return floor, vacuum_pos

def get_single_possible_moves(pos):
    moves = []
    if pos[0] > 0: moves.append("UP")
    if pos[0] < ROW - 1: moves.append("DOWN")
    if pos[1] > 0: moves.append("LEFT")
    if pos[1] < COL - 1: moves.append("RIGHT")
    return moves

def apply_move(floor, vacuum_pos, move):
    if move == "STAY":
        return copy.deepcopy(floor), vacuum_pos
    temp_floor = copy.deepcopy(floor)
    if move == "UP": temp_vac_pos = (vacuum_pos[0] - 1, vacuum_pos[1])
    elif move == "DOWN": temp_vac_pos = (vacuum_pos[0] + 1, vacuum_pos[1])
    elif move == "LEFT": temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] - 1)
    else: temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] + 1)
    
    if temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] == 1:
        temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] = 0
    return temp_floor, temp_vac_pos

def get_n0_of_dirty_tiles(floor: list) -> int:
    """
    Counting the number of dirty tiles
    """
    number_of_dirty_tile = 0
    for row in floor:
        for tile in row:
            if tile == 1:
                number_of_dirty_tile += 1
    return number_of_dirty_tile

def find_dirty_tiles_position(floor) -> list:
    """
    Find all the position (x, y) of dirty tiles on the floor
    """
    positions = []
    for row in range(ROW):
        for col in range(COL):
            if floor[row][col] == 1:
                positions.append((row, col))
    return positions

def min_manhattan_distance(floor, vacuum_pos) -> int:
    """
    Find the nearest dirty tiles
    """
    dirty_tiles = find_dirty_tiles_position(floor)
    if not dirty_tiles:
        return 0
    dis = abs(dirty_tiles[0][0]-vacuum_pos[0]) + abs(dirty_tiles[0][1]-vacuum_pos[1])
    for pos in dirty_tiles:
        temp_dis = abs(pos[0]-vacuum_pos[0]) + abs(pos[1]-vacuum_pos[1])
        if dis > temp_dis:
            dis = temp_dis
    return dis
           
def run_A_Star():
    floor1, vacuum_pos1 = floor_and_vacuumpos_initialize()
    floor2, vacuum_pos2 = floor_and_vacuumpos_initialize()
    frontier = Queue()
    root = Node(floor_state=(floor1, floor2), position=(vacuum_pos1, vacuum_pos2), parent=None, birth_action=None)
    
    # Khởi tạo chi phí
    root.path_cost = 0
    h1 = get_n0_of_dirty_tiles(floor1) + min_manhattan_distance(floor1, vacuum_pos1)
    h2 = get_n0_of_dirty_tiles(floor2) + min_manhattan_distance(floor2, vacuum_pos2)
    root.heuristic_cost = max(h1, h2)
    root.cal_total_cost()
    
    frontier.enqueue(root)
    visited_state = set()
    step = 0
    
    while True:
        if frontier.is_empty():
            print("Frontier is empty!!")
            return None
        
        current_node = frontier.pop_the_highest_priority()
        step += 1
        
        # Dieu kien dung: ca 2 state deu dat GOAL
        if current_node.floor_state[0] == GOAL_STATE and current_node.floor_state[1] == GOAL_STATE:
            print(f"Tong so buoc duyet (Steps popped): {step}")
            return current_node
            
        current_state_key = (current_node.get_tuple_floor_state(), current_node.position)
        if current_state_key in visited_state:
            continue
        visited_state.add(current_state_key)
        
        # Duyet qua cac huong di chuyen (cung loai di chuyen cho ca 2 state)
        for action in ["UP", "DOWN", "LEFT", "RIGHT"]:
            # State 0 move
            if current_node.floor_state[0] == GOAL_STATE:
                m0 = "STAY"
            else:
                moves0 = get_single_possible_moves(current_node.position[0])
                m0 = action if action in moves0 else "STAY"
                
            # State 1 move
            if current_node.floor_state[1] == GOAL_STATE:
                m1 = "STAY"
            else:
                moves1 = get_single_possible_moves(current_node.position[1])
                m1 = action if action in moves1 else "STAY"
                
            if m0 == "STAY" and m1 == "STAY":
                continue
                
                
            child_floor0, child_pos0 = apply_move(current_node.floor_state[0], current_node.position[0], m0)
            child_floor1, child_pos1 = apply_move(current_node.floor_state[1], current_node.position[1], m1)
            
            child_node = Node(
                floor_state=(child_floor0, child_floor1),
                position=(child_pos0, child_pos1),
                parent=current_node,
                birth_action=f"{action} (Robot 1: {m0}, Robot 2: {m1})"
            )
            
            # Tinh chi phi g(n) va h(n)
            child_node.path_cost = current_node.path_cost + 1
            h0 = get_n0_of_dirty_tiles(child_floor0) + min_manhattan_distance(child_floor0, child_pos0)
            h1 = get_n0_of_dirty_tiles(child_floor1) + min_manhattan_distance(child_floor1, child_pos1)
            child_node.heuristic_cost = max(h0, h1)
            child_node.cal_total_cost()
            
            child_state_key = (child_node.get_tuple_floor_state(), child_node.position)
            if child_state_key in visited_state:
                continue
                
            existing_idx = frontier.contain(child_node)
            if existing_idx == -1:
                frontier.enqueue(child_node)
            else:
                existing_node = frontier.queue[existing_idx]
                if child_node.total_cost < existing_node.total_cost:
                    frontier.pop(existing_idx)
                    frontier.enqueue(child_node)

def main():
    result = run_A_Star()
    if isinstance(result, Node):
        print(f"\nDa tim thay dap an (GOAL FOUND)!!!")
        
        # Trich xuat duong di tu Root den Goal
        path = []
        current = result
        while current is not None:
            path.append(current)
            current = current.parent
        path.reverse()
        
        # In chi tiet tung buoc di chuyen
        for step, node in enumerate(path):
            if step == 0:
                print(f"Buoc xuat phat (Initial State):")
            else:
                print(f"Buoc {step} (Step {step}): Hanh dong di chuyen: {node.birth_action}")
            
            print("--- Trang thai State 1 (Floor 1) ---")
            for row in node.floor_state[0]:
                print(row)
            print(f"Vi tri Robot 1: {node.position[0]}")
            
            print("--- Trang thai State 2 (Floor 2) ---")
            for row in node.floor_state[1]:
                print(row)
            print(f"Vi tri Robot 2: {node.position[1]}")
            print("-" * 30)
    else:
        print(f"Khong tim thay dap an: {result}")

if __name__ == "__main__":
    main()